# OCI regional routing

![Direct inference and regional routing](images/inference-paths.svg)

Call your configured model directly with the Responses API, then send the same prompt through a routing profile and see which region handles the request.

In [ ]:
import json

from IPython.display import JSON, display

from demo_utils import create_responses_clients, show_responses_result, show_routing_result

with open("demo_config.json") as config_file:
    config = json.load(config_file)


In [ ]:
direct_client, routed_client, profile_client = create_responses_clients(
    config["oci_profile"], config["direct_region"], config["profile_region"],
    config["compartment_id"],
)


## Prompt

Both calls use the same prompt and generation settings.

In [ ]:
prompt = """Summarize this support incident in two concise sentences, then suggest one next action.
Incident: Customers experienced intermittent checkout failures between 09:00 and 09:12 UTC.
The application team rolled back a recent release at 09:10 UTC. Error rates returned to normal
at 09:12 UTC. The team is reviewing logs to identify the cause."""

response_settings = {
    "input": [{"role": "user", "content": prompt}],
    "max_output_tokens": 4096,
    "temperature": 0.2,
    "stream": False,
    "store": False,
}

## Direct inference

Send the prompt directly to the selected model.


In [ ]:
direct_response = direct_client.responses.with_raw_response.create(
    model=config["model_id"],
    **response_settings,
)
show_responses_result(direct_response)


## Routing profile

Inspect the profile's state, allowed models and allowed regions.


In [ ]:
profile = profile_client.get_routing_profile(config["routing_profile_id"]).data

display(JSON({
    "state": profile.lifecycle_state,
    "allowed_models": profile.model_routing_policy.allowed_models,
    "allowed_regions": profile.region_routing_policy.allowed_regions,
}, root="Routing profile", expanded=True))


## Routed inference

Use the routing-profile OCID as the model ID. The router chooses an allowed OCI region.

`selected_oci_region` comes from the `x-genai-selected-region` response header; `null` means the header was absent. For externally hosted models, the selected OCI region does not identify the model's physical execution location.


In [ ]:
routed_response = routed_client.responses.with_raw_response.create(
    model=config["routing_profile_id"],
    **response_settings,
)
show_responses_result(routed_response)


## Full response

Expand the headers and body to inspect all metadata returned by the routed request.

In [ ]:
display(JSON({
    "headers": dict(routed_response.headers),
    "body": routed_response.parse().model_dump(),
}, root="Routed response", expanded=False))

In [ ]:
show_routing_result(routed_response)